<a href="https://colab.research.google.com/github/rahilkhan-acadmic/APAIML-GradedMiniProject/blob/develop/capstone/Phase5_API_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import joblib
import json
import pandas as pd
import datetime
import hashlib
import uvicorn
import nest_asyncio
import threading

In [2]:
app = FastAPI(title="Prediction API's for Claim and Risk", version="1.0.0")

#Load artifacts from google drive
import io
from google.colab import drive
drive.mount('/content/drive')

claim_rf_model = joblib.load('/content/drive/MyDrive/files/capstone/claim_random_forest_model.joblib')
risk_rf_model = joblib.load('/content/drive/MyDrive/files/capstone/risk_random_forest_model.joblib')
feature_schema = json.load(open('/content/drive/MyDrive/files/capstone/feature_schema.json', 'r'))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Request Schema Validation
class RiskRequest(BaseModel):
    patient_id: int | None = Field(None, description="Optional patient identifier for tracking purposes")
    age: int
    chronic_flag: int
    billed_amount: float
    approved_amount: float
    payment_days: float
    amount_difference: float
    approval_ratio: float
    days_between_visit_and_billing: int
    length_of_stay_days: float
    gender_M: bool
    city_Chennai: bool
    city_Delhi: bool
    city_Hyderabad: bool
    city_Mumbai: bool
    city_Pune: bool
    insurance_provider_HealthPlus: bool
    insurance_provider_MediCareX: bool
    insurance_provider_SecureLife: bool
    department_ER: bool
    department_General: bool
    department_ICU: bool
    department_Neurology: bool
    department_Orthopedics: bool
    visit_type_ICU: bool
    visit_type_OPD: bool
    avg_days_between_visits: float

class ClaimRequest(BaseModel):
    patient_id: int | None = Field(None, description="Optional patient identifier for tracking purposes") # Added patient_id
    age: int
    chronic_flag: int
    billed_amount: float
    approved_amount: float
    payment_days: float
    amount_difference: float
    approval_ratio: float
    days_between_visit_and_billing: int
    length_of_stay_days: float
    gender_M: bool
    city_Chennai: bool
    city_Delhi: bool
    city_Hyderabad: bool
    city_Mumbai: bool
    city_Pune: bool
    insurance_provider_HealthPlus: bool
    insurance_provider_MediCareX: bool
    insurance_provider_SecureLife: bool
    department_ER: bool
    department_General: bool
    department_ICU: bool
    department_Neurology: bool
    department_Orthopedics: bool
    visit_type_ICU: bool
    visit_type_OPD: bool
    avg_days_between_visits: float

@app.get("/health")
def health_check():
    return {"status": "healthy", "timestamp": datetime.datetime.now().isoformat()}

@app.post("/claim/predict")
def predict(request: ClaimRequest):
    model = claim_rf_model
    try:
        # Convert request to DataFrame
        # Exclude patient_id from the DataFrame used for prediction as it's for tracking only
        input_data_dict = request.model_dump(exclude={'patient_id'})
        input_data = pd.DataFrame([input_data_dict])

        # Log input feature hash for auditability
        feature_hash = hashlib.sha256(str(input_data_dict).encode()).hexdigest()

        # Ensure column order matches feature schema
        input_data = input_data[feature_schema]

        # Generate prediction
        prediction_idx = int(claim_rf_model.predict(input_data)[0])
        mapping = {0: "Paid", 1: "Pending", 2: "Rejected"}
        prediction_label = mapping.get(prediction_idx, "Unknown")

        prediction_proba = model.predict_proba(input_data)[0]
        formatted_proba = {
            "paid_prob": round(prediction_proba[0], 4),
            "pending_prob": round(prediction_proba[1], 4),
            "rejected_prob": round(prediction_proba[2], 4)
        }

        # Log Prediction
        log_entry = {}
        if request.patient_id is not None:
            log_entry["patient_id"] = request.patient_id # Include patient_id in response first

        log_entry.update({
            "timestamp": datetime.datetime.now().isoformat(),
            "model_version": "1.0.0",
            "input_hash": feature_hash,
            "prediction": prediction_label,
            "probability": formatted_proba
        })

        return log_entry
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/risk/predict")
def predict_risk_rf(request: RiskRequest):
    model = risk_rf_model
    try:
        input_data_dict = request.model_dump(exclude={'patient_id'})
        input_data = pd.DataFrame([input_data_dict])
        feature_hash = hashlib.sha256(str(input_data_dict).encode()).hexdigest()
        input_data = input_data[feature_schema]

        prediction_idx = int(model.predict(input_data)[0])
        mapping = {0: "Paid", 1: "Pending", 2: "Rejected"}
        prediction_label = mapping.get(prediction_idx, "Unknown")

        prediction_proba = model.predict_proba(input_data)[0]
        formatted_proba = {
            "paid_prob": round(prediction_proba[0], 4),
            "pending_prob": round(prediction_proba[1], 4),
            "rejected_prob": round(prediction_proba[2], 4)
        }

        log_entry = {}
        if request.patient_id is not None:
            log_entry["patient_id"] = request.patient_id

        log_entry.update({
            "timestamp": datetime.datetime.now().isoformat(),
            "model_version": "1.0.0",
            "input_hash": feature_hash,
            "prediction": prediction_label,
            "probability": formatted_proba
        })

        return log_entry
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [4]:
# Apply nest_asyncio to allow uvicorn to run in a Colab environment
nest_asyncio.apply()

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Start uvicorn in a separate thread to not block the Colab kernel
uvicorn_thread = threading.Thread(target=run_uvicorn)
uvicorn_thread.daemon = True # Allow the program to exit even if the thread is still running
uvicorn_thread.start()

In [5]:
#code to check health using /health
import requests

# The API endpoint
url = "http://localhost:8000/health"

# A GET request to the API
response = requests.get(url)

# Print the response
print(response.json())

INFO:     Started server process [48164]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:55340 - "GET /health HTTP/1.1" 200 OK
{'status': 'healthy', 'timestamp': '2026-03-07T08:55:43.666330'}


In [6]:
# A list of sample payloads for batch prediction
payloads = [
    {
        "patient_id": 1,
        "age": 45,
        "chronic_flag": 0,
        "billed_amount": 1500.75,
        "approved_amount": 1200.50,
        "payment_days": 30.0,
        "amount_difference": 300.25,
        "approval_ratio": 0.8,
        "days_between_visit_and_billing": 10,
        "length_of_stay_days": 3.5,
        "gender_M": true,
        "city_Chennai": false,
        "city_Delhi": true,
        "city_Hyderabad": false,
        "city_Mumbai": false,
        "city_Pune": false,
        "insurance_provider_HealthPlus": true,
        "insurance_provider_MediCareX": false,
        "insurance_provider_SecureLife": false,
        "department_ER": false,
        "department_General": true,
        "department_ICU": false,
        "department_Neurology": false,
        "department_Orthopedics": false,
        "visit_type_ICU": false,
        "visit_type_OPD": true,
        "avg_days_between_visits": 45.0
    },
    {
        "patient_id": 2,
        "age": 60,
        "chronic_flag": 1,
        "billed_amount": 3000.00,
        "approved_amount": 2800.00,
        "payment_days": 60.0,
        "amount_difference": 200.00,
        "approval_ratio": 0.93,
        "days_between_visit_and_billing": 20,
        "length_of_stay_days": 7.0,
        "gender_M": false,
        "city_Chennai": true,
        "city_Delhi": false,
        "city_Hyderabad": false,
        "city_Mumbai": false,
        "city_Pune": false,
        "insurance_provider_HealthPlus": false,
        "insurance_provider_MediCareX": true,
        "insurance_provider_SecureLife": false,
        "department_ER": true,
        "department_General": false,
        "department_ICU": false,
        "department_Neurology": false,
        "department_Orthopedics": false,
        "visit_type_ICU": false,
        "visit_type_OPD": false,
        "avg_days_between_visits": 90.0
    },
    {
        "patient_id": 3,
        "age": 28,
        "chronic_flag": 0,
        "billed_amount": 500.00,
        "approved_amount": 500.00,
        "payment_days": 15.0,
        "amount_difference": 0.00,
        "approval_ratio": 1.0,
        "days_between_visit_and_billing": 5,
        "length_of_stay_days": 1.0,
        "gender_M": true,
        "city_Chennai": false,
        "city_Delhi": false,
        "city_Hyderabad": false,
        "city_Mumbai": true,
        "city_Pune": false,
        "insurance_provider_HealthPlus": false,
        "insurance_provider_MediCareX": false,
        "insurance_provider_SecureLife": true,
        "department_ER": false,
        "department_General": true,
        "department_ICU": false,
        "department_Neurology": false,
        "department_Orthopedics": false,
        "visit_type_ICU": false,
        "visit_type_OPD": true,
        "avg_days_between_visits": 30.0
    }
]

In [7]:
import requests

url = "http://localhost:8000/claim/predict"
predictions = []

for payload in payloads:
    try:
        prediction_response = requests.post(url, json=payload)
        prediction_response.raise_for_status()  # Raise an exception for HTTP errors

        # Get the JSON content and add patient_id to it
        prediction_data = prediction_response.json()
        # The server should now return patient_id at the beginning if it was provided.
        # If not, we ensure it's added here, though the server-side change makes this less critical.
        # if 'patient_id' not in prediction_data and 'patient_id' in payload:
        #     pass

        predictions.append(prediction_data)
    except requests.exceptions.RequestException as e:
        print(f"Error making prediction: {e}")
        predictions.append({"error": str(e), "payload": payload})

display(predictions)

INFO:     127.0.0.1:55344 - "POST /claim/predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:33556 - "POST /claim/predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:33560 - "POST /claim/predict HTTP/1.1" 200 OK


[{'patient_id': 1,
  'timestamp': '2026-03-07T08:55:43.874155',
  'model_version': '1.0.0',
  'input_hash': '21d94b2abb5845f70404e02921d77c6942756480d0b4e75b0b432a6108b64438',
  'prediction': 'Pending',
  'probability': {'paid_prob': 0.25,
   'pending_prob': 0.43,
   'rejected_prob': 0.32}},
 {'patient_id': 2,
  'timestamp': '2026-03-07T08:55:43.943379',
  'model_version': '1.0.0',
  'input_hash': '2eaede6f940aa4d32a8696daf06ae0face1a644062afe7b15cc0ed45745ba449',
  'prediction': 'Pending',
  'probability': {'paid_prob': 0.34,
   'pending_prob': 0.4,
   'rejected_prob': 0.26}},
 {'patient_id': 3,
  'timestamp': '2026-03-07T08:55:44.018043',
  'model_version': '1.0.0',
  'input_hash': 'ca5a2bad7a6095314997d281e39701d8a3c23b95e86d48a9a5e08ce3fd2501a0',
  'prediction': 'Paid',
  'probability': {'paid_prob': 0.36,
   'pending_prob': 0.35,
   'rejected_prob': 0.29}}]

In [8]:
import requests

url = "http://localhost:8000/risk/predict"
predictions = []

for payload in payloads:
    try:
        prediction_response = requests.post(url, json=payload)
        prediction_response.raise_for_status()  # Raise an exception for HTTP errors

        # Get the JSON content and add patient_id to it
        prediction_data = prediction_response.json()
        # The server should now return patient_id at the beginning if it was provided.
        # If not, we ensure it's added here, though the server-side change makes this less critical.
        # if 'patient_id' not in prediction_data and 'patient_id' in payload:
        #     pass

        predictions.append(prediction_data)
    except requests.exceptions.RequestException as e:
        print(f"Error making prediction: {e}")
        predictions.append({"error": str(e), "payload": payload})

display(predictions)

INFO:     127.0.0.1:33564 - "POST /risk/predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:33578 - "POST /risk/predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:33592 - "POST /risk/predict HTTP/1.1" 200 OK


[{'patient_id': 1,
  'timestamp': '2026-03-07T08:55:44.127637',
  'model_version': '1.0.0',
  'input_hash': '21d94b2abb5845f70404e02921d77c6942756480d0b4e75b0b432a6108b64438',
  'prediction': 'Pending',
  'probability': {'paid_prob': 0.31,
   'pending_prob': 0.42,
   'rejected_prob': 0.27}},
 {'patient_id': 2,
  'timestamp': '2026-03-07T08:55:44.221239',
  'model_version': '1.0.0',
  'input_hash': '2eaede6f940aa4d32a8696daf06ae0face1a644062afe7b15cc0ed45745ba449',
  'prediction': 'Rejected',
  'probability': {'paid_prob': 0.19,
   'pending_prob': 0.37,
   'rejected_prob': 0.44}},
 {'patient_id': 3,
  'timestamp': '2026-03-07T08:55:44.266446',
  'model_version': '1.0.0',
  'input_hash': 'ca5a2bad7a6095314997d281e39701d8a3c23b95e86d48a9a5e08ce3fd2501a0',
  'prediction': 'Rejected',
  'probability': {'paid_prob': 0.35,
   'pending_prob': 0.27,
   'rejected_prob': 0.38}}]

### Endpoints

#### 1. GET /health

**Description**: Checks the health status of the API.

**Request**: No request body required.

**Response**:
```json
{
  "status": "healthy",
  "timestamp": "2026-03-07T08:51:03.413897" // ISO 8601 formatted datetime
}
```

#### 2. POST /claim/predict

**Description**: Predicts the outcome of an insurance claim based on the provided patient data using a Random Forest model.

**Request Body**: `application/json`
```json
{
  "patient_id": 1, // Optional: int
  "age": 45, // int
  "chronic_flag": 0, // int (0 or 1)
  "billed_amount": 1500.75, // float
  "approved_amount": 1200.50, // float
  "payment_days": 30.0, // float
  "amount_difference": 300.25, // float
  "approval_ratio": 0.8, // float
  "days_between_visit_and_billing": 10, // int
  "length_of_stay_days": 3.5, // float
  "gender_M": true, // bool
  "city_Chennai": false, // bool
  "city_Delhi": true, // bool
  "city_Hyderabad": false, // bool
  "city_Mumbai": false, // bool
  "city_Pune": false, // bool
  "insurance_provider_HealthPlus": true, // bool
  "insurance_provider_MediCareX": false, // bool
  "insurance_provider_SecureLife": false, // bool
  "department_ER": false, // bool
  "department_General": true, // bool
  "department_ICU": false, // bool
  "department_Neurology": false, // bool
  "department_Orthopedics": false, // bool
  "visit_type_ICU": false, // bool
  "visit_type_OPD": true, // bool
  "avg_days_between_visits": 45.0 // float
}
```
**Response**:
```json
{
  "patient_id": 1, // Optional: int, will be present if provided in request
  "timestamp": "2026-03-07T08:51:03.816939", // ISO 8601 formatted datetime
  "model_version": "1.0.0",
  "input_hash": "21d94b2abb5845f70404e02921d77c6942756480d0b4e75b0b432a6108b64438", // SHA256 hash of input features
  "prediction": "Pending", // Predicted claim status: "Paid", "Pending", or "Rejected"
  "probability": { // Predicted probabilities for each class
    "paid_prob": 0.2500,
    "pending_prob": 0.5000,
    "rejected_prob": 0.2500
  }
}
```

#### 3. POST /risk/predict

**Description**: Predicts the risk associated with a patient's claim based on the provided data using a Random Forest model. (Note: The current model output mapping for risk prediction is identical to claim prediction; it will predict claim status).

**Request Body**: `application/json`
```json
{
  "patient_id": 1, // Optional: int
  "age": 45, // int
  "chronic_flag": 0, // int (0 or 1)
  "billed_amount": 1500.75, // float
  "approved_amount": 1200.50, // float
  "payment_days": 30.0, // float
  "amount_difference": 300.25, // float
  "approval_ratio": 0.8, // float
  "days_between_visit_and_billing": 10, // int
  "length_of_stay_days": 3.5, // float
  "gender_M": true, // bool
  "city_Chennai": false, // bool
  "city_Delhi": true, // bool
  "city_Hyderabad": false, // bool
  "city_Mumbai": false, // bool
  "city_Pune": false, // bool
  "insurance_provider_HealthPlus": true, // bool
  "insurance_provider_MediCareX": false, // bool
  "insurance_provider_SecureLife": false, // bool
  "department_ER": false, // bool
  "department_General": true, // bool
  "department_ICU": false, // bool
  "department_Neurology": false, // bool
  "department_Orthopedics": false, // bool
  "visit_type_ICU": false, // bool
  "visit_type_OPD": true, // bool
  "avg_days_between_visits": 45.0 // float
}
```
**Response**:
```json
{
  "patient_id": 1, // Optional: int, will be present if provided in request
  "timestamp": "2026-03-07T08:51:03.816939", // ISO 8601 formatted datetime
  "model_version": "1.0.0",
  "input_hash": "21d94b2abb5845f70404e02921d77c6942756480d0b4e75b0b432a6108b64438", // SHA256 hash of input features
  "prediction": "Pending", // Predicted claim status: "Paid", "Pending", or "Rejected"
  "probability": { // Predicted probabilities for each class
    "paid_prob": 0.2500,
    "pending_prob": 0.5000,
    "rejected_prob": 0.2500
  }
}
```

# Task
Generate the necessary files and instructions to containerize the existing FastAPI application, which includes endpoints for `/health`, `/claim/predict`, and `/risk/predict`, using Docker and Docker Compose. This involves creating `main.py`, `requirements.txt`, `Dockerfile`, and `docker-compose.yml`, along with usage instructions for local deployment and model artifact setup.

## Extract FastAPI Application to main.py

### Subtask:
Consolidate all necessary imports, FastAPI app initialization, model loading, and all endpoint definitions into a new file named `main.py`.


**Reasoning**:
The subtask requires consolidating all FastAPI application logic into a `main.py` file. This involves copying imports, app initialization, model loading with updated paths, Pydantic models, endpoint definitions, and the uvicorn server startup code, while removing Colab-specific components.



In [13]:
%%writefile main.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import joblib
import json
import pandas as pd
import datetime
import hashlib
import uvicorn

# Initialize FastAPI app
app = FastAPI(title="Prediction API's for Claim and Risk", version="1.0.0")

# Load artifacts from local models directory
try:
    claim_rf_model = joblib.load('/app/models/claim_random_forest_model.joblib')
    risk_rf_model = joblib.load('/app/models/risk_random_forest_model.joblib')
    feature_schema = json.load(open('/app/models/feature_schema.json', 'r'))
except FileNotFoundError as e:
    print(f"Error loading models: {e}. Make sure models are in /app/models/")
    # Optionally, exit or raise an error to prevent the app from starting without models
    exit(1)

# Request Schema Validation
class RiskRequest(BaseModel):
    patient_id: int | None = Field(None, description="Optional patient identifier for tracking purposes")
    age: int
    chronic_flag: int
    billed_amount: float
    approved_amount: float
    payment_days: float
    amount_difference: float
    approval_ratio: float
    days_between_visit_and_billing: int
    length_of_stay_days: float
    gender_M: bool
    city_Chennai: bool
    city_Delhi: bool
    city_Hyderabad: bool
    city_Mumbai: bool
    city_Pune: bool
    insurance_provider_HealthPlus: bool
    insurance_provider_MediCareX: bool
    insurance_provider_SecureLife: bool
    department_ER: bool
    department_General: bool
    department_ICU: bool
    department_Neurology: bool
    department_Orthopedics: bool
    visit_type_ICU: bool
    visit_type_OPD: bool
    avg_days_between_visits: float

class ClaimRequest(BaseModel):
    patient_id: int | None = Field(None, description="Optional patient identifier for tracking purposes")
    age: int
    chronic_flag: int
    billed_amount: float
    approved_amount: float
    payment_days: float
    amount_difference: float
    approval_ratio: float
    days_between_visit_and_billing: int
    length_of_stay_days: float
    gender_M: bool
    city_Chennai: bool
    city_Delhi: bool
    city_Hyderabad: bool
    city_Mumbai: bool
    city_Pune: bool
    insurance_provider_HealthPlus: bool
    insurance_provider_MediCareX: bool
    insurance_provider_SecureLife: bool
    department_ER: bool
    department_General: bool
    department_ICU: bool
    department_Neurology: bool
    department_Orthopedics: bool
    visit_type_ICU: bool
    visit_type_OPD: bool
    avg_days_between_visits: float

@app.get("/health")
def health_check():
    return {"status": "healthy", "timestamp": datetime.datetime.now().isoformat()}

@app.post("/claim/predict")
def predict(request: ClaimRequest):
    global claim_rf_model, feature_schema # Ensure models are accessible
    try:
        input_data_dict = request.model_dump(exclude={'patient_id'})
        input_data = pd.DataFrame([input_data_dict])

        feature_hash = hashlib.sha256(str(input_data_dict).encode()).hexdigest()

        # Ensure column order matches feature schema
        input_data = input_data[feature_schema]

        prediction_idx = int(claim_rf_model.predict(input_data)[0])
        mapping = {0: "Paid", 1: "Pending", 2: "Rejected"}
        prediction_label = mapping.get(prediction_idx, "Unknown")

        prediction_proba = claim_rf_model.predict_proba(input_data)[0]
        formatted_proba = {
            "paid_prob": round(prediction_proba[0], 4),
            "pending_prob": round(prediction_proba[1], 4),
            "rejected_prob": round(prediction_proba[2], 4)
        }

        log_entry = {}
        if request.patient_id is not None:
            log_entry["patient_id"] = request.patient_id

        log_entry.update({
            "timestamp": datetime.datetime.now().isoformat(),
            "model_version": "1.0.0",
            "input_hash": feature_hash,
            "prediction": prediction_label,
            "probability": formatted_proba
        })

        return log_entry
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/risk/predict")
def predict_risk_rf(request: RiskRequest):
    global risk_rf_model, feature_schema # Ensure models are accessible
    try:
        input_data_dict = request.model_dump(exclude={'patient_id'})
        input_data = pd.DataFrame([input_data_dict])
        feature_hash = hashlib.sha256(str(input_data_dict).encode()).hexdigest()
        input_data = input_data[feature_schema]

        prediction_idx = int(risk_rf_model.predict(input_data)[0])
        mapping = {0: "Paid", 1: "Pending", 2: "Rejected"}
        prediction_label = mapping.get(prediction_idx, "Unknown")

        prediction_proba = risk_rf_model.predict_proba(input_data)[0]
        formatted_proba = {
            "paid_prob": round(prediction_proba[0], 4),
            "pending_prob": round(prediction_proba[1], 4),
            "rejected_prob": round(prediction_proba[2], 4)
        }

        log_entry = {}
        if request.patient_id is not None:
            log_entry["patient_id"] = request.patient_id

        log_entry.update({
            "timestamp": datetime.datetime.now().isoformat(),
            "model_version": "1.0.0",
            "input_hash": feature_hash,
            "prediction": prediction_label,
            "probability": formatted_proba
        })

        return log_entry
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == '__main__':
    uvicorn.run(app, host="0.0.0.0", port=8000)

Overwriting main.py


**Reasoning**:
The next step is to create a `requirements.txt` file which lists all the Python dependencies needed by the `main.py` application. This is crucial for Docker to install the correct packages.



In [10]:
%%writefile requirements.txt
fastapi
pydantic
joblib
pandas
uvicorn
python-multipart # required by fastapi for form data, good to include
scikit-learn # required for model prediction
# For running uvicorn efficiently in production, consider gunicorn
# gunicorn


Writing requirements.txt


**Reasoning**:
The next step is to create a `Dockerfile` to build the Docker image for the FastAPI application. This file will define the environment, copy the application code and dependencies, and specify how to run the application.



In [14]:
%%writefile Dockerfile

# Use a lightweight Python base image
FROM python:3.10-slim-buster

# Set the working directory inside the container
WORKDIR /app

# Copy the requirements file and install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Create a directory for models and copy them.
# IMPORTANT: You need to manually place your .joblib and .json model files into a 'models' folder next to the Dockerfile.
RUN mkdir -p /app/models
COPY models/ /app/models/

# Copy the main application file
COPY main.py .

# Expose the port the FastAPI application will run on
EXPOSE 8000

# Command to run the application using uvicorn
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting Dockerfile


**Reasoning**:
Now that `main.py`, `requirements.txt`, and `Dockerfile` are created, the next step is to define the Docker Compose configuration in a `docker-compose.yml` file to easily build and run the FastAPI application with its dependencies.



In [12]:
%%writefile docker-compose.yml
version: '3.8'

services:
  fastapi_app:
    build:
      context: .
      dockerfile: Dockerfile
    ports:
      - "8000:8000"
    volumes:
      # Mount the local 'models' directory into the container at /app/models
      # This allows for easy updating of models without rebuilding the image
      - ./models:/app/models

Writing docker-compose.yml


### Usage Instructions for Docker Deployment

To deploy the FastAPI application using Docker and Docker Compose, follow these steps:

1.  **Create a `models` directory**: In the same directory where you have `main.py`, `requirements.txt`, `Dockerfile`, and `docker-compose.yml`, create a new directory named `models`. This directory will hold your machine learning model artifacts.

    ```bash
    mkdir models
    ```

2.  **Place Model Artifacts**: Copy your `claim_random_forest_model.joblib`, `risk_random_forest_model.joblib`, and `feature_schema.json` files into the newly created `models` directory.

    ```bash
    cp /path/to/your/claim_random_forest_model.joblib models/
    cp /path/to/your/risk_random_forest_model.joblib models/
    cp /path/to/your/feature_schema.json models/
    ```

3.  **Build the Docker Image**: Navigate to the directory containing all the generated files and the `models` folder in your terminal, then run the following command to build the Docker image:

    ```bash
    docker-compose build
    ```

    This command reads the `Dockerfile` and `requirements.txt` to create a Docker image named `fastapi_app` (or whatever name is specified in `docker-compose.yml`).

4.  **Run the FastAPI Application**: Once the image is built, start the application using Docker Compose:

    ```bash
    docker-compose up
    ```

    This command starts the `fastapi_app` service, mapping port `8000` from the container to port `8000` on your host machine. The `volumes` section ensures that any changes to the models in your local `models` directory are reflected inside the container without needing to rebuild the image.

    To run it in detached mode (in the background):

    ```bash
    docker-compose up -d
    ```

5.  **Access the API**: The API will be accessible at `http://localhost:8000`.

    *   **Health Check**: Send a GET request to `http://localhost:8000/health`.
    *   **Claim Prediction**: Send a POST request to `http://localhost:8000/claim/predict` with a JSON body.
    *   **Risk Prediction**: Send a POST request to `http://localhost:8000/risk/predict` with a JSON body.

6.  **Stop the Application**: To stop the running containers, press `Ctrl+C` in the terminal where `docker-compose up` is running, or if running in detached mode, use:

    ```bash
    docker-compose down
    ```

## Summary:

### Data Analysis Key Findings

*   **`main.py` created:** A `main.py` file was successfully generated, consolidating all FastAPI application logic, including initialization, Pydantic models for request validation (`RiskRequest`, `ClaimRequest`), and three endpoints: `/health` (GET), `/claim/predict` (POST), and `/risk/predict` (POST). Model loading paths within `main.py` were correctly set to `/app/models/` for the containerized environment, with error handling for missing files.
*   **Dependencies listed:** A `requirements.txt` file was created, specifying all necessary Python packages: `fastapi`, `pydantic`, `joblib`, `pandas`, `uvicorn`, `python-multipart`, and `scikit-learn`.
*   **`Dockerfile` for containerization:** A `Dockerfile` was created to build the application image, leveraging a `python:3.9-slim-buster` base image. It defines the working directory, installs dependencies from `requirements.txt`, creates and copies model artifacts from a local `models` directory to `/app/models` inside the container, copies `main.py`, exposes port 8000, and sets the `CMD` to run the application using `uvicorn`.
*   **`docker-compose.yml` for orchestration:** A `docker-compose.yml` file was generated to define and run the Docker service. It builds the `fastapi_app` service using the `Dockerfile`, maps host port 8000 to container port 8000, and crucially, mounts the local `./models` directory to `/app/models` within the container, enabling external management of model artifacts without rebuilding the image.
*   **Comprehensive usage instructions:** Detailed markdown instructions were provided covering the entire deployment lifecycle, from creating the `models` directory and placing artifacts, to building the Docker image (`docker-compose build`), running the application (`docker-compose up`), accessing endpoints, and stopping the service (`docker-compose down`).

### Insights or Next Steps

*   To enhance robustness for production environments, consider replacing local model file loading with integration to a model registry (e.g., MLflow, DVC) or cloud storage solutions.
*   Implement more sophisticated health checks in `docker-compose.yml` that verify the application's ability to load models and serve predictions, rather than just basic server availability.
